In [ ]:
#
import pandas as pd
import numpy as np

In [ ]:
### cell 0 ###

filename = "/home/dias-benchmarks/new_notebooks/imdb/movie_metadata.csv"

m = pd.read_csv(filename)
factor = 500
# Repeat the rows of m block‐wise using integer‐position indexing rather than concatenating DataFrames
i = list(range(len(m)))
m = m.take(i * factor)

In [ ]:
### cell 1 ###

m.info()

In [ ]:
### cell 2 ###

# Combine all drops into a single call and group numeric conversions
cols_to_drop = [
    "color",
    "director_facebook_likes",
    "actor_3_facebook_likes",
    "actor_2_name",
    "actor_1_facebook_likes",
    "actor_1_name",
    "cast_total_facebook_likes",
    "movie_imdb_link",
    "language",
    "actor_2_facebook_likes",
    "aspect_ratio",
    "actor_3_name",
    "facenumber_in_poster",
    "plot_keywords",
    "country",
]
cols_to_convert = ["duration", "budget", "gross", "imdb_score"]

# Drop unwanted columns first to reduce frame size
m.drop(columns=cols_to_drop, inplace=True)
# Convert the numeric columns in one vectorized operation
m[cols_to_convert] = m[cols_to_convert].apply(pd.to_numeric)
# Strip and set the index in one step
m.set_index(m.movie_title.str.strip(), inplace=True)

In [ ]:
### cell 3 ###

m.columns

In [ ]:
### cell 4 ###

ninety_min_num_movies = len(m.duration[m.duration > 90.0])
total_num_movies = len(m.index[m.duration != np.nan])
ninety_min_num_movies / total_num_movies

In [ ]:
### cell 5 ###

# Use a boolean mask and sum() instead of slicing and len()
two_hour_movies = (m.duration > 120.0).sum()
two_hour_movies / total_num_movies

In [ ]:
### cell 6 ###

# Compute counts via vectorized boolean sums (no intermediate Series slices)
num_movies_directed = m["director_name"].notna().sum()
spiel = m["director_name"].eq("Steven Spielberg").sum()
spiel / num_movies_directed

In [ ]:
### cell 7 ###

# Optimized code for cell 8
e_movies = m.loc[m["director_name"].eq("Clint Eastwood")]
# `NaN < value` is False, so explicit notna() checks aren’t needed
e_gross_under_budget = e_movies["gross"].lt(e_movies["budget"]).sum()
e_gross_under_budget / len(e_movies)

In [ ]:
### cell 8 ###

# Filter out rows with null gross or budget once
movies_with_budget_and_gross = m.dropna(subset=["gross", "budget"])

# From the cleaned DataFrame, select those with gross > budget
gross_over_budget = movies_with_budget_and_gross.loc[
    movies_with_budget_and_gross.gross > movies_with_budget_and_gross.budget
]

# Compute the ratio
len(gross_over_budget) / len(movies_with_budget_and_gross)

In [ ]:
### cell 9 ###

gross = m["gross"]
average_gross = gross.mean()
notna_mask = gross.notna()
movie_grossed_over_average = ((gross > average_gross) & notna_mask).sum()
total_movie_with_gross = notna_mask.sum()
movie_grossed_over_average / total_movie_with_gross

In [ ]:
### cell 10 ###

movies_with_scores = m[["imdb_score", "gross", "budget"]].dropna()
mask = movies_with_scores.imdb_score > 6
((movies_with_scores.gross < movies_with_scores.budget) & mask).sum() / mask.sum()

In [ ]:
### cell 11 ###

mask = movies_with_scores.imdb_score <= 6
((movies_with_scores.gross > movies_with_scores.budget) & mask).sum() / mask.sum()

In [ ]:
### cell 12 ###

scores = m.imdb_score

In [ ]:
### cell 13 ###

scores.describe()

In [ ]:
### cell 14 ###

mean = scores.mean()
median = scores.quantile(0.5)
std = scores.std()
mean, median, std